# 07. Verify A7+06 blend `model.ts` on DLC validation

목적: DLC validation 104개 전체에서 제출용 TorchScript의 `P(RERECORDED)`와

`p_expected = (1/3) * p_A7 + (2/3) * p_06`

를 sample-level로 비교한다.

중요 조건:
- 제출 `inference.py`와 동일한 FP32 / no autocast
- 16 frames / stride 2 / center clip
- 384 resize + center crop
- ImageNet normalization
- threshold = 0.5

확인 항목:
- probability absolute difference
- threshold 0.5 label flip
- DLC Macro-F1 변화
- 차이가 큰 sample
- 결과 CSV / JSON 저장

In [1]:
# 1. Colab / Drive setup
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import time

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

cv2.setNumThreads(1)
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'CUDA GPU에서 실행하세요.'

print('device:', DEVICE)
print('gpu   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

Mounted at /content/drive
device: cuda
gpu   : NVIDIA L4
torch : 2.8.0+cu126


In [2]:
# 2. Paths / knobs
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
DATASET_ROOT = DRIVE_PROJECT_ROOT / 'DATASET'
DLC_ROOT = DATASET_ROOT / 'DLC-2021'
DLC_SPLIT_CSV = DLC_ROOT / 'dlc_split.csv'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs' / 'stage1'

BLEND_DIR = OUTPUT_ROOT / 'a7_06_probblend_2to1' / 'vjepa2_1_b_blend'
MODEL_TS = BLEND_DIR / 'model.ts'
A7_PRED_CSV = OUTPUT_ROOT / 'dlc' / 'vjepa2_1_b' / 'val_predictions.csv'
ADAPTED_06_PRED_CSV = OUTPUT_ROOT / 'dlc_ccd_synrr_200' / 'vjepa2_1_b' / 'val_predictions.csv'

ALPHA_06 = 2.0 / 3.0
THRESHOLD = 0.5
VIDEO_EXT = {'.mp4','.avi','.mov','.mkv','.m4v','.3gp','.3gpp','.wmv','.webm'}

print('DLC ROOT  :', DLC_ROOT, '| exists:', DLC_ROOT.is_dir())
print('DLC SPLIT :', DLC_SPLIT_CSV, '| exists:', DLC_SPLIT_CSV.is_file())
print('MODEL_TS  :', MODEL_TS, '| exists:', MODEL_TS.is_file())
print('A7 CSV    :', A7_PRED_CSV, '| exists:', A7_PRED_CSV.is_file())
print('06 CSV    :', ADAPTED_06_PRED_CSV, '| exists:', ADAPTED_06_PRED_CSV.is_file())

assert DLC_ROOT.is_dir(), f'DLC root not found: {DLC_ROOT}'
assert DLC_SPLIT_CSV.is_file(), f'DLC split CSV not found: {DLC_SPLIT_CSV}'
assert MODEL_TS.is_file(), f'model.ts not found: {MODEL_TS}'
assert A7_PRED_CSV.is_file(), f'A7 prediction CSV not found: {A7_PRED_CSV}'
assert ADAPTED_06_PRED_CSV.is_file(), f'06 prediction CSV not found: {ADAPTED_06_PRED_CSV}'


DLC ROOT  : /content/drive/MyDrive/Blackbox-Detection/DATASET/DLC-2021 | exists: True
DLC SPLIT : /content/drive/MyDrive/Blackbox-Detection/DATASET/DLC-2021/dlc_split.csv | exists: True
MODEL_TS  : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/a7_06_probblend_2to1/vjepa2_1_b_blend/model.ts | exists: True
A7 CSV    : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/val_predictions.csv | exists: True
06 CSV    : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc_ccd_synrr_200/vjepa2_1_b/val_predictions.csv | exists: True


## CSV sanity check
A7/06 CSV를 `video_id` 기준으로 합쳐 expected probability blend를 만든다.
104개가 정확히 merge되지 않으면 중단한다.

In [3]:
# 3. Load A7 / 06 prediction tables and build expected blend
a7 = pd.read_csv(A7_PRED_CSV)
m06 = pd.read_csv(ADAPTED_06_PRED_CSV)

required = {'video_id','label','prob_rerecorded'}
for name, frame in [('A7', a7), ('06', m06)]:
    missing = required - set(frame.columns)
    assert not missing, f'{name} CSV missing columns: {sorted(missing)}'
    assert frame['video_id'].is_unique, f'{name} video_id is not unique'

wide = a7[['video_id','label','prob_rerecorded']].rename(columns={'prob_rerecorded':'p_a7'}).merge(
    m06[['video_id','label','prob_rerecorded']].rename(columns={'label':'label_06','prob_rerecorded':'p_06'}),
    on='video_id', how='inner', validate='one_to_one'
)
assert (wide['label'] == wide['label_06']).all(), 'A7/06 labels disagree'
wide = wide.drop(columns='label_06').copy()
wide['p_expected'] = (1.0 - ALPHA_06) * wide['p_a7'] + ALPHA_06 * wide['p_06']
wide['expected_pred'] = np.where(wide['p_expected'] >= THRESHOLD, 'RERECORDED', 'ORIGINAL')

print('A7 rows       :', len(a7))
print('06 rows       :', len(m06))
print('merged rows   :', len(wide))
print('class balance :', wide['label'].value_counts().to_dict())
assert len(wide) == 104, f'Expected 104 DLC validation videos, got {len(wide)}'
display(wide.head())

A7 rows       : 104
06 rows       : 104
merged rows   : 104
class balance : {'RERECORDED': 60, 'ORIGINAL': 44}


,video_id,label,p_a7,p_06,p_expected,expected_pred
0,dlc__aze_passport__00.or0001,ORIGINAL,0.000001,0.000005,0.000003,ORIGINAL
1,dlc__aze_passport__00.or0002,ORIGINAL,0.000002,0.000005,0.000004,ORIGINAL
2,dlc__aze_passport__00.re0001,RERECORDED,0.999934,0.999453,0.999613,RERECORDED
3,dlc__aze_passport__00.re0002,RERECORDED,0.999872,0.999855,0.999861,RERECORDED
4,dlc__aze_passport__00.re0003,RERECORDED,0.999722,0.999920,0.999854,RERECORDED


In [4]:
# 4. Macro-F1 helper
LABELS = ('ORIGINAL','RERECORDED')
def macro_f1(y_true, y_pred):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); scores=[]
    for cls in LABELS:
        tp=np.sum((y_true==cls)&(y_pred==cls))
        fp=np.sum((y_true!=cls)&(y_pred==cls))
        fn=np.sum((y_true==cls)&(y_pred!=cls))
        denom=2*tp+fp+fn
        scores.append(0.0 if denom==0 else (2.0*tp)/denom)
    return float(np.mean(scores))

for prob_col,name in [('p_a7','A7'),('p_06','06'),('p_expected','expected blend')]:
    pred=np.where(wide[prob_col].to_numpy()>=THRESHOLD,'RERECORDED','ORIGINAL')
    print(f'{name:14s} Macro-F1 @0.5 = {macro_f1(wide["label"],pred):.12f}')

A7             Macro-F1 @0.5 = 1.000000000000
06             Macro-F1 @0.5 = 0.990180341800
expected blend Macro-F1 @0.5 = 1.000000000000


## Resolve DLC paths
기존 Stage 1 notebooks와 동일하게 실제 DLC 영상은 `DLC-2021/or/clips_video/**`, `DLC-2021/re/clips_video/**`에서 index한다.


In [6]:
# 5. Resolve every CSV video_id to one DLC file
# Same resolver logic as existing Stage 1 notebooks (03/05).

def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")


def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    """
    Index:
        DLC-2021/or/clips_video/**
        DLC-2021/re/clips_video/**

    key:
        (source, relative_path_without_suffix)

    Example:
        ("or", "aze_passport/00.or0001")
    """
    index = {}
    source_counts = {}

    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"

        if not clips_root.is_dir():
            raise FileNotFoundError(
                f"DLC clips directory not found: {clips_root}"
            )

        count = 0

        for path in clips_root.rglob("*"):
            if (
                not path.is_file()
                or path.suffix.lower() not in VIDEO_EXT
            ):
                continue

            rel_no_suffix = (
                path.relative_to(clips_root)
                .with_suffix("")
                .as_posix()
            )

            key = (
                source,
                _normalize_rel_text(rel_no_suffix),
            )

            if key in index and index[key] != str(path):
                raise ValueError(
                    "Duplicate DLC relative video key detected: "
                    f"{key} -> {index[key]} and {path}"
                )

            index[key] = str(path)
            count += 1

        source_counts[source] = count

    print("indexed DLC videos:", source_counts)

    assert source_counts == {
        "or": 290,
        "re": 400,
    }, source_counts

    return index


def _resolve_dlc_video(
    index: dict[tuple[str, str], str],
    source: str,
    clip_id: str,
) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)

    # 1. Exact match
    exact_key = (source, clip_id)

    if exact_key in index:
        return index[exact_key]

    # 2. Extra folder level fallback
    matches = []

    for (indexed_source, rel_key), video_path in index.items():
        if indexed_source != source:
            continue

        if (
            rel_key.startswith(clip_id + "/")
            or rel_key.endswith("/" + clip_id)
            or rel_key == clip_id
        ):
            matches.append(video_path)

    if len(matches) == 1:
        return matches[0]

    # 3. Leaf fallback, only if unique inside source
    leaf = Path(clip_id).name

    leaf_matches = [
        video_path
        for (indexed_source, rel_key), video_path in index.items()
        if (
            indexed_source == source
            and Path(rel_key).name == leaf
        )
    ]

    if len(leaf_matches) == 1:
        return leaf_matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve "
        f"clip_id={clip_id!r}, source={source!r}. "
        f"exact={exact_key in index}, "
        f"prefix/suffix matches={len(matches)}, "
        f"leaf matches={len(leaf_matches)}"
    )


# ------------------------------------------------------------------
# Build exact video index
# ------------------------------------------------------------------

DLC_VIDEO_INDEX = _build_dlc_video_index()


# ------------------------------------------------------------------
# Read authoritative DLC validation metadata
# ------------------------------------------------------------------

split_df = pd.read_csv(DLC_SPLIT_CSV).copy()

required = {
    "clip_id",
    "source",
    "split",
}

missing = required - set(split_df.columns)

assert not missing, (
    f"dlc_split.csv missing columns: {sorted(missing)}"
)

split_df["split"] = (
    split_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

split_df["source"] = (
    split_df["source"]
    .astype(str)
    .str.strip()
    .str.lower()
)

val_meta = split_df.loc[
    split_df["split"].eq("val"),
    [
        "clip_id",
        "source",
    ],
].copy()

val_meta["video_id"] = (
    "dlc__"
    + val_meta["clip_id"]
    .astype(str)
    .str.replace("/", "__", regex=False)
)

assert len(val_meta) == 104, (
    f"Expected 104 DLC val rows, got {len(val_meta)}"
)

assert val_meta["video_id"].is_unique


# ------------------------------------------------------------------
# Attach clip_id/source to A7+06 prediction table
# ------------------------------------------------------------------

# 셀 재실행에 안전하게
wide = wide.drop(
    columns=[
        col
        for col in ("clip_id", "source", "video_path")
        if col in wide.columns
    ]
)

wide = wide.merge(
    val_meta[
        [
            "video_id",
            "clip_id",
            "source",
        ]
    ],
    on="video_id",
    how="left",
    validate="one_to_one",
)

assert wide["clip_id"].notna().all(), (
    "Some prediction video_ids were not found in dlc_split.csv"
)

assert wide["source"].notna().all()


# ------------------------------------------------------------------
# Resolve actual video paths
# ------------------------------------------------------------------

wide["video_path"] = [
    _resolve_dlc_video(
        DLC_VIDEO_INDEX,
        source,
        clip_id,
    )
    for source, clip_id in zip(
        wide["source"],
        wide["clip_id"],
    )
]

assert len(wide) == 104

assert wide["video_path"].nunique() == 104, (
    "Resolved paths are not unique."
)

assert all(
    Path(path).is_file()
    for path in wide["video_path"]
), "Some resolved video paths do not exist."


print("[PASS] all 104 DLC validation videos resolved uniquely")

display(
    wide[
        [
            "video_id",
            "source",
            "clip_id",
            "video_path",
        ]
    ].head(10)
)

indexed DLC videos: {'or': 290, 're': 400}
[PASS] all 104 DLC validation videos resolved uniquely


,video_id,source,clip_id,video_path
0,dlc__aze_passport__00.or0001,or,aze_passport/00.or0001,/content/drive/MyDrive/Blackbox-Detection/DATA...
1,dlc__aze_passport__00.or0002,or,aze_passport/00.or0002,/content/drive/MyDrive/Blackbox-Detection/DATA...
2,dlc__aze_passport__00.re0001,re,aze_passport/00.re0001,/content/drive/MyDrive/Blackbox-Detection/DATA...
3,dlc__aze_passport__00.re0002,re,aze_passport/00.re0002,/content/drive/MyDrive/Blackbox-Detection/DATA...
4,dlc__aze_passport__00.re0003,re,aze_passport/00.re0003,/content/drive/MyDrive/Blackbox-Detection/DATA...
5,dlc__aze_passport__00.re0004,re,aze_passport/00.re0004,/content/drive/MyDrive/Blackbox-Detection/DATA...
6,dlc__aze_passport__00.re0005,re,aze_passport/00.re0005,/content/drive/MyDrive/Blackbox-Detection/DATA...
7,dlc__aze_passport__00.re0006,re,aze_passport/00.re0006,/content/drive/MyDrive/Blackbox-Detection/DATA...
8,dlc__esp_id__03.or0001,or,esp_id/03.or0001,/content/drive/MyDrive/Blackbox-Detection/DATA...
9,dlc__esp_id__03.or0002,or,esp_id/03.or0002,/content/drive/MyDrive/Blackbox-Detection/DATA...


## Submission-equivalent preprocessing
제출 `inference.py`와 동일하게 FP32 / no autocast로 실행한다.

In [7]:
# 6. Exact submission-equivalent preprocessing
S1_NUM_FRAMES=16; S1_STRIDE=2; S1_CROP_SIZE=384
S1_SHORT_SIDE=int(round(S1_CROP_SIZE*256.0/224.0))
S1_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32)[:,None,None,None]
S1_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32)[:,None,None,None]

def total_frames(path):
    cap=cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened(): raise RuntimeError(f'cannot open {path}')
        total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    finally: cap.release()
    return max(total,1)

def center_clip_indices(total):
    total=max(int(total),1); span=(S1_NUM_FRAMES-1)*S1_STRIDE+1; max_start=max(total-span,0); start=max_start//2
    return np.clip(start+np.arange(S1_NUM_FRAMES,dtype=np.int64)*S1_STRIDE,0,total-1)

def decode_frames(path,frame_indices):
    wanted=[int(x) for x in np.asarray(frame_indices).reshape(-1)]
    order=np.argsort(wanted,kind='stable'); sorted_wanted=[wanted[i] for i in order]
    cap=cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened(): raise RuntimeError(f'cannot open {path}')
        decoded=[None]*len(sorted_wanted); cap.set(cv2.CAP_PROP_POS_FRAMES,sorted_wanted[0]); position=sorted_wanted[0]; last=None
        for slot,target in enumerate(sorted_wanted):
            if last is not None and target<position: decoded[slot]=last; continue
            frame=None; ok=False
            while position<=target:
                ok,frame=cap.read(); position+=1
                if not ok: break
            if not ok or frame is None: decoded[slot]=last; continue
            last=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); decoded[slot]=last
        if all(x is None for x in decoded): raise RuntimeError(f'cannot decode {path}')
        first=next(x for x in decoded if x is not None); filled=[x if x is not None else first for x in decoded]
    finally: cap.release()
    restored=[None]*len(wanted)
    for slot,orig in enumerate(order): restored[int(orig)]=filled[slot]
    return np.stack(restored)

def resize_shorter_side(frame,target):
    h,w=frame.shape[:2]
    if min(h,w)==target: return frame
    scale=target/float(min(h,w)); nw=max(target,int(round(w*scale))); nh=max(target,int(round(h*scale)))
    interp=cv2.INTER_AREA if scale<1.0 else cv2.INTER_LINEAR
    return cv2.resize(frame,(nw,nh),interpolation=interp)

def preprocess_video(path):
    frames=decode_frames(path,center_clip_indices(total_frames(path)))
    processed=np.empty((S1_NUM_FRAMES,S1_CROP_SIZE,S1_CROP_SIZE,3),dtype=np.uint8)
    for i,frame in enumerate(frames):
        resized=resize_shorter_side(frame,S1_SHORT_SIDE); h,w=resized.shape[:2]
        top=max((h-S1_CROP_SIZE)//2,0); left=max((w-S1_CROP_SIZE)//2,0)
        processed[i]=resized[top:top+S1_CROP_SIZE,left:left+S1_CROP_SIZE]
    clip=torch.from_numpy(np.ascontiguousarray(processed)).float().div_(255.0).permute(3,0,1,2).contiguous()
    clip=(clip-S1_MEAN)/S1_STD
    return clip.unsqueeze(0)

In [8]:
# 7. Load model.ts and warm up with one real clip
model_ts=torch.jit.load(str(MODEL_TS),map_location=DEVICE).eval()
sample_path=Path(wide.iloc[0]['video_path'])
sample_clip=preprocess_video(sample_path).to(device=DEVICE,dtype=torch.float32)
with torch.inference_mode(): sample_logits=model_ts(sample_clip)
if isinstance(sample_logits,(tuple,list)): sample_logits=sample_logits[0]
print('sample output shape:',tuple(sample_logits.shape)); print('sample logits:',sample_logits.detach().cpu())
assert tuple(sample_logits.shape)==(1,2); assert torch.isfinite(sample_logits).all()
del sample_clip,sample_logits; torch.cuda.empty_cache()
print('[PASS] model.ts warmup')

sample output shape: (1, 2)
sample logits: tensor([[-3.5509e-06, -1.2548e+01]])
[PASS] model.ts warmup


## Run all 104 videos
의도적으로 batch=1, FP32, no autocast로 실행한다.

In [9]:
# 8. model.ts inference on all 104 DLC videos
p_ts=[]; elapsed_each=[]; failures=[]
torch.cuda.synchronize(); total_started=time.perf_counter()
with torch.inference_mode():
    for row in tqdm(wide.itertuples(index=False),total=len(wide),desc='model.ts DLC'):
        path=Path(row.video_path)
        try:
            started=time.perf_counter()
            clip=preprocess_video(path).to(device=DEVICE,dtype=torch.float32,non_blocking=True)
            logits=model_ts(clip)
            if isinstance(logits,(tuple,list)): logits=logits[0]
            assert tuple(logits.shape)==(1,2); assert torch.isfinite(logits).all()
            prob=float(torch.softmax(logits.float(),dim=1)[0,1].cpu().item())
            torch.cuda.synchronize(); elapsed=time.perf_counter()-started
            p_ts.append(prob); elapsed_each.append(elapsed); del clip,logits
        except Exception as e:
            p_ts.append(np.nan); elapsed_each.append(np.nan); failures.append({'video_id':row.video_id,'video_path':str(path),'error':repr(e)})
total_elapsed=time.perf_counter()-total_started
print('failures:',len(failures)); print('total sec:',total_elapsed); print('sec/video:',total_elapsed/len(wide))
if failures:
    display(pd.DataFrame(failures)); raise RuntimeError('model.ts inference failures detected')
wide['p_model_ts']=p_ts; wide['model_ts_elapsed_sec']=elapsed_each
print('[PASS] all 104 videos inferred')

model.ts DLC:   0%|          | 0/104 [00:00<?, ?it/s]

failures: 0
total sec: 548.943323165
sec/video: 5.278301184278846
[PASS] all 104 videos inferred


In [10]:
# 9. Sample-level comparison + summary
wide['abs_diff']=np.abs(wide['p_model_ts']-wide['p_expected'])
wide['signed_diff']=wide['p_model_ts']-wide['p_expected']
wide['model_ts_pred']=np.where(wide['p_model_ts']>=THRESHOLD,'RERECORDED','ORIGINAL')
wide['label_flip_vs_expected']=wide['model_ts_pred']!=wide['expected_pred']
wide['expected_margin_to_0.5']=np.abs(wide['p_expected']-THRESHOLD)
wide['model_ts_margin_to_0.5']=np.abs(wide['p_model_ts']-THRESHOLD)

diff=wide['abs_diff'].to_numpy(float)
summary={
 'num_videos':int(len(wide)),'alpha_06':float(ALPHA_06),'threshold':float(THRESHOLD),
 'a7_macro_f1_at_0.5':macro_f1(wide['label'],np.where(wide['p_a7']>=THRESHOLD,'RERECORDED','ORIGINAL')),
 'adapted_06_macro_f1_at_0.5':macro_f1(wide['label'],np.where(wide['p_06']>=THRESHOLD,'RERECORDED','ORIGINAL')),
 'expected_blend_macro_f1_at_0.5':macro_f1(wide['label'],wide['expected_pred']),
 'model_ts_macro_f1_at_0.5':macro_f1(wide['label'],wide['model_ts_pred']),
 'mean_abs_diff':float(np.mean(diff)),'median_abs_diff':float(np.median(diff)),
 'p90_abs_diff':float(np.quantile(diff,.90)),'p95_abs_diff':float(np.quantile(diff,.95)),
 'p99_abs_diff':float(np.quantile(diff,.99)),'max_abs_diff':float(np.max(diff)),
 'num_label_flips_vs_expected':int(wide['label_flip_vs_expected'].sum()),
 'total_inference_seconds':float(total_elapsed),'mean_inference_seconds':float(np.nanmean(wide['model_ts_elapsed_sec']))
}
print(json.dumps(summary,indent=2,ensure_ascii=False))

{
  "num_videos": 104,
  "alpha_06": 0.6666666666666666,
  "threshold": 0.5,
  "a7_macro_f1_at_0.5": 1.0,
  "adapted_06_macro_f1_at_0.5": 0.9901803417996412,
  "expected_blend_macro_f1_at_0.5": 1.0,
  "model_ts_macro_f1_at_0.5": 1.0,
  "mean_abs_diff": 0.0001110533737606678,
  "median_abs_diff": 3.774960835034591e-07,
  "p90_abs_diff": 1.578132311510495e-05,
  "p95_abs_diff": 0.00012506691273302372,
  "p99_abs_diff": 0.0041122911373773765,
  "max_abs_diff": 0.006114787422120516,
  "num_label_flips_vs_expected": 0,
  "total_inference_seconds": 548.943323165,
  "mean_inference_seconds": 5.277429664211551
}


In [11]:
# 10. Largest probability differences
largest=wide.sort_values('abs_diff',ascending=False)[[
 'video_id','label','p_a7','p_06','p_expected','p_model_ts','signed_diff','abs_diff',
 'expected_pred','model_ts_pred','label_flip_vs_expected'
]].head(20)
display(largest)

,video_id,label,p_a7,p_06,p_expected,p_model_ts,signed_diff,abs_diff,expected_pred,model_ts_pred,label_flip_vs_expected
37,dlc__grc_passport__05.or0003,ORIGINAL,0.008544,0.373347,0.251746,0.245632,-0.006115,0.006115,ORIGINAL,ORIGINAL,False
102,dlc__svk_id__07.re0003,RERECORDED,0.850469,0.371906,0.531427,0.535656,0.004229,0.004229,RERECORDED,RERECORDED,False
100,dlc__svk_id__07.re0001,RERECORDED,0.990660,0.972336,0.978444,0.978785,0.000341,0.000341,RERECORDED,RERECORDED,False
97,dlc__svk_id__06.re0004,RERECORDED,0.975484,0.999942,0.991789,0.992036,0.000246,0.000246,RERECORDED,RERECORDED,False
56,dlc__lva_passport__02.re0002,RERECORDED,0.996990,0.989632,0.992084,0.992217,0.000132,0.000132,RERECORDED,RERECORDED,False
80,dlc__rus_internalpassport__01.re0005,RERECORDED,0.999555,0.996937,0.997809,0.997680,-0.000130,0.000130,RERECORDED,RERECORDED,False
38,dlc__grc_passport__05.or0004,ORIGINAL,0.000536,0.006388,0.004437,0.004536,0.000099,0.000099,ORIGINAL,ORIGINAL,False
103,dlc__svk_id__07.re0004,RERECORDED,0.998235,0.959837,0.972637,0.972561,-0.000075,0.000075,RERECORDED,RERECORDED,False
60,dlc__lva_passport__02.re0006,RERECORDED,0.999876,0.996931,0.997912,0.997881,-0.000031,0.000031,RERECORDED,RERECORDED,False
70,dlc__lva_passport__04.re0003,RERECORDED,0.999737,0.998916,0.999190,0.999172,-0.000017,0.000017,RERECORDED,RERECORDED,False


In [12]:
# 11. Threshold flips
flips=wide.loc[wide['label_flip_vs_expected']].sort_values('expected_margin_to_0.5')[[
 'video_id','label','p_a7','p_06','p_expected','p_model_ts','abs_diff',
 'expected_pred','model_ts_pred','expected_margin_to_0.5'
]]
print('threshold flips:',len(flips)); display(flips)

threshold flips: 0


,video_id,label,p_a7,p_06,p_expected,p_model_ts,abs_diff,expected_pred,model_ts_pred,expected_margin_to_0.5


In [13]:
# 12. Classification correctness changes
wide['expected_correct']=wide['expected_pred']==wide['label']
wide['model_ts_correct']=wide['model_ts_pred']==wide['label']
error_change=wide.loc[wide['expected_correct']!=wide['model_ts_correct']][[
 'video_id','label','p_expected','p_model_ts','abs_diff','expected_pred','model_ts_pred','expected_correct','model_ts_correct'
]].sort_values('abs_diff',ascending=False)
print('samples whose correctness changed:',len(error_change)); display(error_change)

samples whose correctness changed: 0


,video_id,label,p_expected,p_model_ts,abs_diff,expected_pred,model_ts_pred,expected_correct,model_ts_correct


## Interpretation

- **label flip = 0이고 model.ts F1 == expected blend F1**: 제출 TorchScript/FP32가 DLC 의사결정을 망가뜨린 증거는 없다. public 하락은 synthetic-domain generalization 실패 쪽이 더 유력하다.
- **label flip이 있거나 DLC F1이 1.0에서 하락**: FP32/AMP 차이 또는 TorchScript export path가 실제 결정을 바꿨다. 먼저 export/inference를 조사한다.
- probability drift가 있어도 label flip이 0이면 DLC에는 영향이 없지만, hidden의 borderline sample에는 영향을 줄 가능성은 남는다.

In [14]:
# 13. Automatic verdict
expected_f1=summary['expected_blend_macro_f1_at_0.5']; ts_f1=summary['model_ts_macro_f1_at_0.5']; n_flips=summary['num_label_flips_vs_expected']; max_diff=summary['max_abs_diff']
print('='*72); print('VERDICT'); print('='*72)
if n_flips==0 and abs(expected_f1-ts_f1)<1e-12:
    print('[PASS] model.ts has exactly the same threshold-0.5 decisions as the CSV blend on all 104 DLC videos.')
    print('=> Public-score drop is unlikely to be caused by a DLC-visible TorchScript/FP32 decision bug.')
    if max_diff>1e-3:
        print(f'[NOTE] max probability drift={max_diff:.6g}. No DLC label flip, but hidden borderline samples may still be sensitive.')
else:
    print('[ALERT] model.ts changes at least one DLC decision or Macro-F1.')
    print(f'expected F1={expected_f1:.12f}, model.ts F1={ts_f1:.12f}, flips={n_flips}')
    print('=> Investigate FP32 vs AMP / TorchScript export before blaming domain shift entirely.')
print('='*72)

VERDICT
[PASS] model.ts has exactly the same threshold-0.5 decisions as the CSV blend on all 104 DLC videos.
=> Public-score drop is unlikely to be caused by a DLC-visible TorchScript/FP32 decision bug.
[NOTE] max probability drift=0.00611479. No DLC label flip, but hidden borderline samples may still be sensitive.


In [15]:
# 14. Save diagnostics next to model.ts
OUT_CSV=BLEND_DIR/'model_ts_vs_csv_dlc_predictions.csv'
OUT_JSON=BLEND_DIR/'model_ts_vs_csv_dlc_summary.json'
save_columns=[
 'video_id','label','video_path','p_a7','p_06','p_expected','p_model_ts','signed_diff','abs_diff',
 'expected_pred','model_ts_pred','label_flip_vs_expected','expected_correct','model_ts_correct',
 'expected_margin_to_0.5','model_ts_margin_to_0.5','model_ts_elapsed_sec'
]
wide[save_columns].to_csv(OUT_CSV,index=False)
with OUT_JSON.open('w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
print('saved:',OUT_CSV); print('saved:',OUT_JSON)

saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/a7_06_probblend_2to1/vjepa2_1_b_blend/model_ts_vs_csv_dlc_predictions.csv
saved: /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/a7_06_probblend_2to1/vjepa2_1_b_blend/model_ts_vs_csv_dlc_summary.json
